# ADNI Left Hippocampus split creation

This notebook reads subject IDs from the ADNI metadata CSV and splits left hippocampus OBJ scans into train/val/test by subject, so each subject's scans stay in a single split.

Output directory (relative to the notebook's working directory):
`../examples/splits/splits_left_hippocampus_ADNI`


In [1]:
from pathlib import Path
import csv
import json
import random
import re

labels_csv = Path('/home/jakaria/starmen/starman_all_meshes/subsets/first_100_ids_10_scans/scan_labels_output_random_noacc.csv')
obj_dir = Path('/home/jakaria/starmen/starman_all_meshes/subsets/first_100_ids_10_scans/meshes/output_random_noacc/mesh_minimal_obj')
output_dir = Path('../examples/splits/splits_starmen')

train_ratio = 0.80
test_ratio = 0.15
val_ratio = 0.05
random_seed = 42

assert abs(train_ratio + test_ratio + val_ratio - 1.0) < 1e-6
assert labels_csv.exists(), f'Missing CSV labels file: {labels_csv}'
assert obj_dir.exists(), f'Missing OBJ directory: {obj_dir}'
output_dir.mkdir(parents=True, exist_ok=True)


In [2]:
scan_to_subject = {}
subjects_from_csv = set()

with labels_csv.open(newline='') as f:
    reader = csv.DictReader(f)
    required_columns = {'subject_id', 'scan_name'}
    assert required_columns.issubset(set(reader.fieldnames or [])), (
        f'CSV must contain columns {required_columns}, got {reader.fieldnames}'
    )

    for row in reader:
        scan_name = (row.get('scan_name') or '').strip()
        sid_raw = (row.get('subject_id') or '').strip()

        if not scan_name or not sid_raw:
            continue

        sid = str(int(sid_raw))
        existing_sid = scan_to_subject.get(scan_name)
        if existing_sid is not None:
            assert existing_sid == sid, (
                f'Conflicting subject IDs for scan {scan_name}: {existing_sid} vs {sid}'
            )
        else:
            scan_to_subject[scan_name] = sid

        subjects_from_csv.add(sid)

print(f'CSV scans: {len(scan_to_subject)}')
print(f'CSV subjects: {len(subjects_from_csv)}')


CSV scans: 1000
CSV subjects: 100


In [3]:
obj_files = sorted(p.name for p in obj_dir.glob('*.obj'))

scan_pattern = re.compile(r'__scan-(.+)\.obj$')
sid_pattern = re.compile(r'__sid-(\d+)__')
subject_pattern = re.compile(r'__subject_s(\d+)__')

subject_to_files = {}
unmatched_files = []
unknown_scan_files = []
id_mismatch_files = []

for fname in obj_files:
    scan_match = scan_pattern.search(fname)
    if not scan_match:
        unmatched_files.append(fname)
        continue

    scan_name = scan_match.group(1)
    csv_sid = scan_to_subject.get(scan_name)
    if csv_sid is None:
        unknown_scan_files.append(fname)
        continue

    parsed_ids = []

    sid_match = sid_pattern.search(fname)
    if sid_match:
        parsed_ids.append(str(int(sid_match.group(1))))

    subject_match = subject_pattern.search(fname)
    if subject_match:
        parsed_ids.append(str(int(subject_match.group(1))))

    if not parsed_ids:
        id_mismatch_files.append((fname, csv_sid, 'missing_id_patterns'))
        continue

    if any(parsed_sid != csv_sid for parsed_sid in parsed_ids):
        id_mismatch_files.append((fname, csv_sid, ','.join(parsed_ids)))
        continue

    subject_to_files.setdefault(csv_sid, []).append(fname)

subjects = sorted(subject_to_files.keys(), key=int)

print(f'OBJ files: {len(obj_files)}')
print(f'Subjects with OBJ files: {len(subjects)}')
print(f'Unmatched files (pattern): {len(unmatched_files)}')
print(f'Files with scan missing in CSV: {len(unknown_scan_files)}')
print(f'Files with ID mismatch: {len(id_mismatch_files)}')


OBJ files: 1000
Subjects with OBJ files: 100
Unmatched files (pattern): 0
Files with scan missing in CSV: 0
Files with ID mismatch: 0


In [4]:
rng = random.Random(random_seed)
rng.shuffle(subjects)

num_subjects = len(subjects)
num_train = int(num_subjects * train_ratio)
num_test = int(num_subjects * test_ratio)
num_val = num_subjects - num_train - num_test

train_subjects = subjects[:num_train]
test_subjects = subjects[num_train:num_train + num_test]
val_subjects = subjects[num_train + num_test:]

def collect_files(subject_list):
    files = []
    for sid in subject_list:
        files.extend(subject_to_files[sid])
    return sorted(files)

train_files = collect_files(train_subjects)
test_files = collect_files(test_subjects)
val_files = collect_files(val_subjects)

train_path = output_dir / 'train_split_starmen.json'
test_path = output_dir / 'test_split_starmen.json'
val_path = output_dir / 'val_split_starmen.json'

with train_path.open('w') as f:
    json.dump(train_files, f, indent=2)
with test_path.open('w') as f:
    json.dump(test_files, f, indent=2)
with val_path.open('w') as f:
    json.dump(val_files, f, indent=2)

print('Wrote:', train_path, test_path, val_path)
print(f'Train subjects/files: {len(train_subjects)}/{len(train_files)}')
print(f'Test subjects/files: {len(test_subjects)}/{len(test_files)}')
print(f'Val subjects/files: {len(val_subjects)}/{len(val_files)}')
print(f'Subject ratios: train {len(train_subjects) / num_subjects:.2%}, '
      f'test {len(test_subjects) / num_subjects:.2%}, '
      f'val {len(val_subjects) / num_subjects:.2%}')


Wrote: ../examples/splits/splits_starmen/train_split_starmen.json ../examples/splits/splits_starmen/test_split_starmen.json ../examples/splits/splits_starmen/val_split_starmen.json
Train subjects/files: 80/800
Test subjects/files: 15/150
Val subjects/files: 5/50
Subject ratios: train 80.00%, test 15.00%, val 5.00%


In [5]:
train_subjects_set = set(train_subjects)
test_subjects_set = set(test_subjects)
val_subjects_set = set(val_subjects)

assert train_subjects_set.isdisjoint(test_subjects_set)
assert train_subjects_set.isdisjoint(val_subjects_set)
assert test_subjects_set.isdisjoint(val_subjects_set)
assert train_subjects_set | test_subjects_set | val_subjects_set == set(subjects)

train_files_set = set(train_files)
test_files_set = set(test_files)
val_files_set = set(val_files)

assert train_files_set.isdisjoint(test_files_set)
assert train_files_set.isdisjoint(val_files_set)
assert test_files_set.isdisjoint(val_files_set)

all_split_files = train_files_set | test_files_set | val_files_set
all_subject_files = set()
for files in subject_to_files.values():
    all_subject_files.update(files)

assert all_split_files == all_subject_files

subject_to_split = {}
for sid in train_subjects:
    subject_to_split[sid] = 'train'
for sid in test_subjects:
    subject_to_split[sid] = 'test'
for sid in val_subjects:
    subject_to_split[sid] = 'val'

for sid, files in subject_to_files.items():
    split = subject_to_split[sid]
    if split == 'train':
        assert set(files).issubset(train_files_set)
    elif split == 'test':
        assert set(files).issubset(test_files_set)
    else:
        assert set(files).issubset(val_files_set)

print('Split checks passed: each subject ID is in exactly one split file.')


Split checks passed: each subject ID is in exactly one split file.
